# Task 1 — Repository discovery

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` and the pinned local clone |
| Command | `bash scripts/clone_repo.sh`; `bash scripts/run_stage2_evidence.sh` |
| Dataset commit | `41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2` |
| Build behavior | Offline; Pages renders this committed executed output. |
```

## Approach and reasoning

The clone command uses `--depth 1` to minimize download size, then pins the
Moodle-selected `huggingface/datasets` revision. Discovery records both every
`.py` file and the parser-selected production files, so exclusions remain
auditable rather than silently changing the reported total.

## Key implementation excerpts

The book embeds the real source-of-truth files; it does not duplicate them.

```{literalinclude} ../scripts/clone_repo.sh
:language: bash
:lines: 1-26
:caption: Shallow clone and pinned revision
```

```{literalinclude} ../parser_service/discover.py
:language: python
:pyobject: discover_python_files
:caption: Deterministic Python file discovery
```

## Executed evidence


In [1]:
from pathlib import Path
import json, sys

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/stage2_manifest.json").exists()
)
sys.path.insert(0, str(ROOT))

from parser_service.discover import discover_python_files

repo = ROOT / "data/datasets"
manifest = json.loads((ROOT / "screenshots/stage2_manifest.json").read_text())
all_python = sorted(repo.rglob("*.py"))
selected = discover_python_files(repo)

print("dataset_commit:", manifest["run"]["dataset_commit"])
print("all_python_files:", len(all_python))
print("parser_selected_files:", len(selected))
print("excluded_or_out_of_scope:", len(all_python) - len(selected))
print("sample:", [p.relative_to(repo).as_posix() for p in selected[:5]])

assert len(manifest["run"]["dataset_commit"]) == 40
assert len(selected) == manifest["run"]["files_processed"] == 138
assert len(all_python) == 233


dataset_commit: 41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2
all_python_files: 233
parser_selected_files: 138
excluded_or_out_of_scope: 95
sample: ['src/datasets/__init__.py', 'src/datasets/arrow_dataset.py', 'src/datasets/arrow_reader.py', 'src/datasets/arrow_writer.py', 'src/datasets/builder.py']


## What this chapter proves

| Requirement | Evidence |
|---|---|
| Shallow clone | The embedded command uses `git clone --depth 1`. |
| File discovery | The executed cell reports 233 total Python files and 138 selected files. |
| Provenance | The exact 40-character dataset commit is printed and asserted. |

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Shallow clone plus deterministic discovery kept the input reproducible. |
| Failed | Earlier evidence recorded `commit_sha=unknown`. |
| Resolution | The runbook now resolves and validates the pinned commit before parsing. |
```
